# 01 — X-IIoTID Dataset Analysis

**Objective:** Check the dataset structure, label balance and missing values.

**Input:** Original `X-IIoTID dataset.csv` (read-only).

**Output:** A short summary, one graph and a JSON report.

**Labels:** `class3` = Normal/Attack; `class2` = broad category; `class1` = attack type. **Labels are not training features.**

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "x_iiotid" / "X-IIoTID dataset.csv"
REPORTS = ROOT / "reports" / "audits"
FIGURES = ROOT / "reports" / "figures"

CORE19 = [
    "Protocol", "Service", "Duration", "Scr_bytes", "Des_bytes", "Conn_state",
    "is_syn_only", "Is_SYN_ACK", "is_pure_ack", "is_with_payload", "FIN or RST",
    "Scr_pkts", "Scr_ip_bytes", "Des_pkts", "Des_ip_bytes", "total_bytes",
    "total_packet", "paket_rate", "byte_rate"
]
LABELS = ["class1", "class2", "class3"]
MISSING_TOKENS = {"", "-", "?", "na", "n/a", "nan", "none", "null", "missing", "unknown"}
CHUNK_SIZE = 30000

if not DATA.is_file():
    raise FileNotFoundError(f"Original dataset not found: {DATA}")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print("Using:", DATA)


## 1. Dataset structure

Check the total columns and confirm the 19 candidate network features and three labels exist.

In [ ]:
# Read only the header; do not load all 820k+ records into memory.
columns = pd.read_csv(DATA, nrows=0).columns.tolist()
if len(set(columns)) != len(columns):
    raise ValueError("Duplicate CSV column names: review the source header")
missing_required = [c for c in CORE19 + LABELS if c not in columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print(f"Total columns: {len(columns)}")
print(f"Selected network features: {len(CORE19)}")
print("Target: class3 (Normal / Attack)")
display(pd.DataFrame({"network_feature": CORE19}))


## 2. Class balance and data quality

Count every record. Missing-value counts include common empty/placeholder tokens, which still need checking before modeling.

In [ ]:
label_counts = {name: Counter() for name in LABELS}
missing_counts = Counter()
row_count = 0

for chunk in pd.read_csv(
    DATA, usecols=CORE19 + LABELS, dtype=str, keep_default_na=False,
    chunksize=CHUNK_SIZE, low_memory=False
):
    row_count += len(chunk)
    for label in LABELS:
        label_counts[label].update(chunk[label].astype(str).str.strip().value_counts().to_dict())
    for feature in CORE19:
        raw = chunk[feature].astype(str).str.strip().str.casefold()
        missing_counts[feature] += int(raw.isin(MISSING_TOKENS).sum())

binary_counts = label_counts["class3"]
if set(binary_counts) != {"Normal", "Attack"}:
    raise ValueError(f"Unexpected class3 labels: {dict(binary_counts)}")
if sum(binary_counts.values()) != row_count:
    raise ValueError("Binary label counts do not match total rows")

balance = pd.DataFrame([
    {"class": name, "records": int(binary_counts[name]),
     "percentage": 100 * binary_counts[name] / row_count}
    for name in ("Normal", "Attack")
])
print(f"Dataset: {row_count:,} records × {len(columns)} columns")
display(balance.round(2))

quality = pd.DataFrame([
    {"feature": name, "missing_or_placeholder": missing_counts[name],
     "percentage": round(100 * missing_counts[name] / row_count, 2)}
    for name in CORE19
]).sort_values("percentage", ascending=False, ignore_index=True)
print("Top 8 candidate features with missing/placeholder values:")
display(quality.head(8))
print("Class levels: class2 =", len(label_counts["class2"]),
      "; class1 =", len(label_counts["class1"]))


## 3. Class balance graph

This graph shows how many Normal and Attack records are available. It is not a model-performance result.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(balance["class"], balance["records"])
for bar, row in zip(bars, balance.to_dict("records")):
    ax.annotate(f"{row['records']:,}  ({row['percentage']:.2f}%)",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center")
ax.set_ylim(0, balance["records"].max() * 1.2)
ax.set_ylabel("Number of records")
ax.set_title("X-IIoTID — Normal vs Attack")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
figure_path = FIGURES / "x_iiotid_v3_class_balance.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", figure_path)


## 4. Save the dataset summary

Keep the original CSV unchanged. Save only summary numbers and the checksum; not raw data.

In [ ]:
checksum = hashlib.sha256()
with DATA.open("rb") as file:
    for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
        checksum.update(block)

report = {
    "dataset": "X-IIoTID", "source_file": DATA.name,
    "source_sha256": checksum.hexdigest(),
    "rows": row_count, "columns": len(columns), "column_names": columns,
    "core19_features": CORE19, "target": "class3",
    "label_columns_excluded_from_features": LABELS,
    "label_counts": {name: dict(counts) for name, counts in label_counts.items()},
    "candidate_feature_missing_or_placeholder_counts": dict(missing_counts),
    "notes": [
        "Original CSV was not modified.",
        "class2 and class1 are labels, not model inputs.",
        "This is an exploratory data audit, not evidence of gateway feature availability.",
        "Duplicate/signature-leakage checks and feature justification belong in Notebook 02."
    ]
}
report_path = REPORTS / "x_iiotid_v3_dataset_analysis.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", report_path)
print("Next: 02_feature_selection.ipynb")
